In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

from utils import HumanEvalTask, MBPPTask
from typing import List, Optional
import os
import json
import numpy as np
import pickle

os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# 0. Prepare Features and Labels

Code Evaluation

In [ ]:
def read_dataset(data_file: str = None,) -> List:
    """
    Reads a dataset and returns a list of tasks.
    """
    dataset = []
    with open(data_file, 'r') as f:
        for line in f:
            dataset.append(
                json.loads(line)
            )
    return dataset

In [ ]:
def evaluate_code(model_name, dataset_name):
    gen_path = f'output/{model_name}/{dataset_name}/generation.jsonl'
    gen_data = read_dataset(gen_path)

    new_line = 'qwen' in model_name.lower()
    if dataset_name == 'humaneval':
        task = HumanEvalTask(new_line=new_line)
    elif dataset_name == 'mbpp':
        task = MBPPTask(new_line=new_line)
    else:
        raise NotImplementedError

    dataset = task.get_dataset()
    n_tasks = len(dataset)
    n_gens = len(gen_data) // n_tasks
    print (f'#tasks: {n_tasks}, #gens: {n_gens}')
    references = [task.get_reference(dataset[i]) for i in range(n_tasks)]
    offset = 0
    if dataset_name == 'mbpp':
        offset = 11

    generations: List[List[Optional[str]]] = [[None] * n_gens for _ in range(n_tasks)]
    for ex in gen_data:
        task_id = ex['task_id']
        if dataset_name == "humaneval":
            idx = int(task_id.split('/')[-1])
        else:
            idx = int(task_id) - offset
        generations[idx][ex['gen_num']] = ex['generation']

    results, results_full = task.process_results(generations, references, return_errs=True)
    print (results)

    passed = {}
    for task_id in results_full:
        for r in results_full[task_id]:
            passed[(r[-1]['task_id'] + offset, r[-1]['completion_id'])] = int(r[-1]['passed'])
    passed = dict(sorted(passed.items(), key=lambda x: x[0]))

    label_path = gen_path.replace('generation.jsonl', 'labels.pickle')
    with open(label_path, 'wb') as f:
        pickle.dump(passed, f)

In [ ]:
evaluate_code(model_name='deepseek-coder-6.7b-base', dataset_name='humaneval')

In [ ]:
evaluate_code(model_name='deepseek-coder-6.7b-base', dataset_name='mbpp')

Load Features and Labels

In [ ]:
def load_feature(model_name, dataset_name, feat_type):
    feat_dir = f'output/{model_name}/{dataset_name}/{feat_type}'
    feat_dict = {}
    for fn in sorted(os.listdir(feat_dir)):
        if '.npy' in fn:
            if dataset_name == 'humaneval':
                task_id = int(fn.split('_')[3])
            elif dataset_name == 'mbpp':
                task_id = int(fn.split('_')[2])
            else:
                raise NotImplementedError
            gen_num = int(fn.split('_')[-1].split('.npy')[0])
            feat_dict[(task_id, gen_num)] = np.load(os.path.join(feat_dir, fn))
    return feat_dict

In [ ]:
def load_prompt_gen_lens(model_name, dataset_name):
    gen_path = f'output/{model_name}/{dataset_name}/generation.jsonl'
    gen_data = read_dataset(gen_path)

    prompt_lens, gen_lens = {}, {}
    for gen_ex in gen_data:
        if dataset_name == "humaneval":
            task_id = int(gen_ex['task_id'].split('/')[-1])
        elif dataset_name == "mbpp":
            task_id = int(gen_ex['task_id'])
        else:
            raise NotImplementedError
        gen_num = int(gen_ex['gen_num'])
        prompt_len = gen_ex['prompt_len']
        out_len = gen_ex['gen_len']
        prompt_lens[(task_id, gen_num)] = prompt_len
        gen_lens[(task_id, gen_num)] = out_len - prompt_len

    prompt_lens = dict(sorted(prompt_lens.items(), key=lambda x: x[0]))
    gen_lens = dict(sorted(gen_lens.items(), key=lambda x: x[0]))
    return prompt_lens, gen_lens

In [ ]:
def load_features_labels(model_name, dataset_name):
    prompt_lens, gen_lens = load_prompt_gen_lens(model_name, dataset_name)
    
    features_dict = {
        'diag_prompt': load_feature(model_name, dataset_name, feat_type='diag_prompt'),
        'diag_response': load_feature(model_name, dataset_name, feat_type='diag_response'),
        'mtd0_kind0': load_feature(model_name, dataset_name, feat_type='mtd0_kind0'),
        'mtd1_kind0': load_feature(model_name, dataset_name, feat_type='mtd1_kind0'),
        'mtd0_kind1': load_feature(model_name, dataset_name, feat_type='mtd0_kind1'),
        'mtd1_kind1': load_feature(model_name, dataset_name, feat_type='mtd1_kind1'),
    }
    
    y_dict = pickle.load(open(f'output/{model_name}/{dataset_name}/labels.pickle', 'rb'))

    X_dict = {}
    for feat_name, feats in features_dict.items():
        for key in feats:
            if key not in X_dict:
                X_dict[key] = []
            if feat_name == 'diag_prompt':
                X_dict[key].extend(feats[key].flatten()/ prompt_lens[key])
            elif feat_name == 'diag_response':
                X_dict[key].extend(feats[key].flatten()/ max(1, gen_lens[key]))
            elif feat_name == 'mtd0_kind0':
                X_dict[key].extend(feats[key].flatten()/ max(1, gen_lens[key]))
            elif feat_name == 'mtd1_kind0':
                X_dict[key].extend(feats[key].flatten()/ max(1, gen_lens[key]))
            elif feat_name == 'mtd0_kind1':
                X_dict[key].extend(feats[key].flatten()/ prompt_lens[key])
            elif feat_name == 'mtd1_kind1':
                X_dict[key].extend(feats[key].flatten()/ prompt_lens[key])

    X_dict = dict(sorted(X_dict.items(), key=lambda x: x[0]))
    y_dict = dict(sorted(y_dict.items(), key=lambda x: x[0]))
    X, y, groups = [], [], []
    for key in X_dict:
        groups.append(key[0])
        X.append(X_dict[key])
        y.append(y_dict[key])
    X = np.array(X)
    y = np.array(y)
    groups = np.array(groups)
    return X, y, groups

# 1. Hallucination Detection

In [ ]:
def train_classifier(X_train, y_train, X_test, y_test):
    print (X_train.shape, X_test.shape)
    print (y_train.shape, y_test.shape)
    bst = xgb.XGBClassifier(tree_method="hist")
    bst.fit(X_train, y_train)
    test_proba = bst.predict_proba(X_test)
    rocauc = roc_auc_score(y_test, test_proba[:, 1])
    return bst, rocauc

In [ ]:
X, y, groups = load_features_labels(model_name='deepseek-coder-6.7b-base', dataset_name='humaneval')
sgkf = StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)
clfs, roc_auc_scores = [], []
for i, (train_index, test_index) in enumerate(sgkf.split(X, y, groups)):
    print ('RUN {}'.format(i))
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    bst, rocauc = train_classifier(X_train, y_train, X_test, y_test)
    clfs.append(bst)
    roc_auc_scores.append(rocauc)

print ('\nROC AUC: ', np.mean(roc_auc_scores).round(3), np.std(roc_auc_scores).round(3))
print (np.array(roc_auc_scores).round(3))

In [ ]:
X, y, groups = load_features_labels(model_name='deepseek-coder-6.7b-base', dataset_name='mbpp')
sgkf = StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)
clfs, roc_auc_scores = [], []
for i, (train_index, test_index) in enumerate(sgkf.split(X, y, groups)):
    print ('RUN {}'.format(i))
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    bst, rocauc = train_classifier(X_train, y_train, X_test, y_test)
    clfs.append(bst)
    roc_auc_scores.append(rocauc)

print ('\nROC AUC: ', np.mean(roc_auc_scores).round(3), np.std(roc_auc_scores).round(3))
print (np.array(roc_auc_scores).round(3))

# 2. Cross-benchmark Transfer

HE $\to$ MBPP transfer

In [ ]:
X_train, y_train, _ = load_features_labels(model_name='deepseek-coder-6.7b-base', dataset_name='humaneval')
X_test, y_test, _ = load_features_labels(model_name='deepseek-coder-6.7b-base', dataset_name='mbpp')

bst, rocauc = train_classifier(X_train, y_train, X_test, y_test)

print ('\nROC AUC: ', rocauc)

MBPP $\to$ HE transfer

In [ ]:
X_train, y_train, _ = load_features_labels(model_name='deepseek-coder-6.7b-base', dataset_name='mbpp')
X_test, y_test, _ = load_features_labels(model_name='deepseek-coder-6.7b-base', dataset_name='humaneval')

bst, rocauc = train_classifier(X_train, y_train, X_test, y_test)

print ('\nROC AUC: ', rocauc)